# Calibration of spectra
`author` Julien Morin <julien.morin@umontpellier.fr>

`date` 15 Nov 2023

References: *Handbook of CCD Astronomy* §6 ; *Handbook of Astronomical Spectroscopy* §6
* flat field correction
* wavelength calibration
* save spectra as FITS files

In [ ]:
# import and settings

import copy
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import astropy.io.fits as pf
from astropy.modeling import models
from astropy.stats import mad_std
from astropy.table import QTable
from astropy import units as u
from astropy.visualization import ZScaleInterval
import ccdproc as ccdp
import specutils
import specutils.fitting as spf
from specutils.manipulation import extract_region
from specutils import Spectrum1D, SpectralRegion

## Load previously computed data 

In [ ]:
# define CCD characteristics
ccd_gain = 0.37 * u.electron / u.adu
ccd_ron = 6.0 * u.electron

In [ ]:
# load spectra
sci_s1d = ccdp.CCDData.read('reduced/Tetha LEO_Object_10s_20190612_233400-1_s1d.fit')
mflat_s1d = ccdp.CCDData.read('reference/master_flat_s1d.fits')
mwcal_s1d = ccdp.CCDData.read('reference/master_wcal_s1d.fits')

## Example 1: normalise flat field and correct other frames

In [ ]:
# normalise flat field to keep only small-scale variations
## perform high-degree polynomial fit
x_ccd = np.linspace(0, mflat_s1d.data.shape[0],  mflat_s1d.data.shape[0])
mflat_coeff = np.polyfit(x_ccd, mflat_s1d.data, deg=11)
## compute polynomial model
mflat_fit = np.poly1d(mflat_coeff)(x_ccd)
## normalise the flat field w/ this model
mflat_n_s1d = mflat_s1d.data / mflat_fit

# plot fit and result
%matplotlib notebook
fig = plt.figure(figsize=(10,8))
#
ax1 = fig.add_subplot(211)
ax1.plot(mflat_s1d.data, c='tab:blue')
ax1.plot(mflat_fit, c='tab:orange')
ax1.axvline(850, ls='--', c='k', lw=0.4)
ax1.set_title('Master flat field and polynomial fit')
#
ax2 = fig.add_subplot(212)
ax2.plot(mflat_n_s1d, c='tab:green')
ax2.axvline(850, ls='--', c='k', lw=0.4)
ax2.set_title('normalised master flat field')
#
plt.show()

In [ ]:
### TODO:
###  - compute fit RMS
###  - test and compare various degrees of the polynomial fit

## Example 2: Wavelength calibration

In [ ]:
# convert wcal spectrum to a specutils.Spectrum1D instance
mwcal_s1d = specutils.Spectrum1D(mwcal_s1d.data*u.adu)

In [ ]:
# detect spectral lines
lines = spf.find_lines_derivative(mwcal_s1d, flux_threshold=7.e4)
lines[lines['line_type'] == 'emission']
print("%d emission lines identified" % len(lines))
print(lines)
# plot identified lines
plt.figure()
plt.plot(mwcal_s1d.flux)
for l in lines:
    plt.axvline(l['line_center_index'], ls='--', c='grey', lw=0.5)
plt.show()

In [ ]:
# fit each line w/ a gaussian model
wcal_amp = []
wcal_x = []
wcal_std = []
plt.figure(figsize=(10,6))
plt.plot(sci_s1d.data*3.e5/np.mean(sci_s1d.data), c='k', lw=0.5)
plt.plot(mwcal_s1d.flux)
print('Amplitude   x0     sigma')
for l in lines:
    sub_region = SpectralRegion((l['line_center_index']-4)*u.pix, (l['line_center_index']+4)*u.pix)
    sub_spectrum = extract_region(mwcal_s1d, sub_region)
    g = spf.estimate_line_parameters(sub_spectrum, models.Gaussian1D())
    wcal_amp.append(g.amplitude.value)
    wcal_x.append(g.mean.value)
    wcal_std.append(g.stddev.value)
    xx = np.linspace((l['line_center_index']-5)*u.pix,(l['line_center_index']+5)*u.pix,22)
    plt.plot(xx, g(xx))
    print("%9.1f   %5.1f  %.2f"%(g.amplitude.value, g.mean.value, g.stddev.value))
plt.show()

In [ ]:
# associate observed lines to tabulated H/Ne/Ar lines and compute preliminary solution
wcal_lam = np.zeros_like(wcal_x)
wcal_lam[24] = 5852.49
wcal_lam[11] = 6562.85 # H alpha
#TODO: compute coeff a and b for linear solution lam = a*x + b
#a = 
#b = 
#print('a = ', a*u.AA, ' ; b = ', b*u.AA)
#lam0 = a*mwcal_s1d.spectral_axis.value + b

In [ ]:
# Re-plot spectrum w/ this first solution
plt.figure(figsize=(10,6))
plt.plot(lam0, sci_s1d.data*3.e5/np.mean(sci_s1d.data), c='k', lw=0.5)
plt.plot(lam0, mwcal_s1d.flux)
for i, x in enumerate(wcal_x):
    if wcal_lam[i] > 1.:
        plt.axvline(a*x+b, ls='--', c='tab:green', lw=1.)
    elif wcal_lam[i] < 0.:
        plt.axvline(a*x+b, ls='--', c='tab:red', lw=0.5)
    else:
        plt.axvline(a*x+b, ls='--', c='grey', lw=0.5)
plt.xlim(3500., 8000.)
plt.show()

In [ ]:
### Further work:
###  - identify more lines
###  - fit a higher degree polynomial wavelength solution
###  - plot residuals and reject possible outliers

## Apply wavelength calibration to science spectrum

In [ ]:
# Save spectrum as a fits file
## primary HDU contains header from parent file
prihdu_sci_fw = pf.PrimaryHDU(header=sci_s1d.meta)
# extension HDU contains data stored as astropy.table.QTable
sci_fw_tab = QTable([lam0[::-1]*u.AA, sci_s1d.data[::-1]*u.electron], names=('wavelength', 'flux'))
sci_fw_hdu = pf.table_to_hdu(sci_fw_tab)
hdul = pf.HDUList([prihdu_sci_fw, sci_fw_hdu])
hdul.writeto('reduced/Tetha LEO_Object_10s_20190612_233400-1_w_s1d.fit', overwrite=True)
# alternatively one can just save the QTable but original header info is lost
#sci_fw_tab.write('reduced/Tetha LEO_Object_10s_20190612_233400-1_fw_s1d.fit', overwrite=True)

In [ ]:
# First method to load only binary table data
spec1d = QTable.read('reduced/Tetha LEO_Object_10s_20190612_233400-1_w_s1d.fit')
print(spec1d)

In [ ]:
# Second method to retrieve metadata info as well
spec1d = pf.open('reduced/Tetha LEO_Object_10s_20190612_233400-1_w_s1d.fit', ignore_missing_end=True)
print(spec1d.info())
print('\n')
print(spec1d[0].header)
print('\n')
print(spec1d[1].header)
print('\n')
print(spec1d[1].data)